# Telco Customer Churn - Exploratory Data Analysis & Modeling Prototyping
This notebook contains the exploratory data analysis (EDA), feature engineering validation, and baseline modeling prototypes for predicting customer churn using the Telco Customer Churn dataset.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set seaborn style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Dataset
First, we load the raw customer data from Kaggle.

In [ ]:
# Path to raw data
raw_data_path = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
if os.path.exists(raw_data_path):
    df = pd.read_csv(raw_data_path)
    print(f"Dataset shape: {df.shape}")
    display(df.head())
else:
    print("Raw CSV file not found. Make sure to download and place it under data/raw/.")

## 2. Basic Inspection & Data Cleaning
We inspect the dataset columns, types, and check for missing values. A known issue in this dataset is whitespace strings in the `TotalCharges` column.

In [ ]:
if os.path.exists(raw_data_path):
    print("Data Types:")
    print(df.dtypes)
    
    # Check for empty string spaces in TotalCharges
    num_spaces = (df['TotalCharges'] == ' ').sum()
    print(f"\nNumber of whitespace values in TotalCharges: {num_spaces}")

We clean `TotalCharges` by replacing whitespace with NaN, converting to float, and filling NaN values with 0.0 since these correspond to new customers (tenure = 0).

In [ ]:
if os.path.exists(raw_data_path):
    df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0.0)
    print("TotalCharges successfully cleaned and converted.")

## 3. Exploratory Data Analysis (EDA)
Let's visualize the target label `Churn` distribution and check for class imbalance.

In [ ]:
if os.path.exists(raw_data_path):
    churn_counts = df['Churn'].value_counts()
    plt.figure(figsize=(6, 4))
    sns.barplot(x=churn_counts.index, y=churn_counts.values, palette='viridis')
    plt.title('Churn Label Distribution')
    plt.ylabel('Count')
    plt.xlabel('Churn')
    plt.show()
    
    churn_rate = (df['Churn'] == 'Yes').mean() * 100
    print(f"Overall Churn Rate: {churn_rate:.2f}%")

### 3.1 Tenure vs Churn
Let's see how the length of the customer relationship (tenure) affects the probability of churning.

In [ ]:
if os.path.exists(raw_data_path):
    plt.figure(figsize=(10, 6))
    sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, common_norm=False, palette='crest')
    plt.title('Tenure Distribution by Churn Status')
    plt.xlabel('Tenure (months)')
    plt.show()

### 3.2 Contract Type vs Churn
Let's analyze how the type of contract (Month-to-month, One year, Two year) correlates with customer loss.

In [ ]:
if os.path.exists(raw_data_path):
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x='Contract', hue='Churn', palette='magma')
    plt.title('Churn Count by Contract Type')
    plt.ylabel('Number of Customers')
    plt.show()

## 4. Feature Engineering
We will engineer 3 new features:
1. `NumServices`: Count of active services subscribed (out of Phone, MultipleLines, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies)
2. `AvgMonthlySpend`: Average spending computed as `TotalCharges / tenure` (or `MonthlyCharges` if tenure is 0)
3. `IsNewCustomer`: 1 if tenure <= 6 months else 0

In [ ]:
if os.path.exists(raw_data_path):
    service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    df['NumServices'] = df[service_cols].apply(lambda row: sum(row == 'Yes'), axis=1)
    
    df['AvgMonthlySpend'] = np.where(df['tenure'] == 0, df['MonthlyCharges'], df['TotalCharges'] / df['tenure'])
    df['IsNewCustomer'] = (df['tenure'] <= 6).astype(int)
    
    print("New features preview:")
    display(df[['tenure', 'MonthlyCharges', 'TotalCharges', 'NumServices', 'AvgMonthlySpend', 'IsNewCustomer']].head())

## 5. Correlation of Numerical Features
Let's plot the correlation matrix for the numerical features including our newly engineered features.

In [ ]:
if os.path.exists(raw_data_path):
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'NumServices', 'AvgMonthlySpend', 'IsNewCustomer']
    corr = df[num_cols].corr()
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Correlation Matrix of Numerical Features')
    plt.show()

## 6. Prototyping Models
We'll prepare the data by encoding categories using `LabelEncoder`, splitting train/test (80/20), applying SMOTE to balance the classes, and evaluating Logistic Regression, Random Forest, and XGBoost classifiers.

In [ ]:
if os.path.exists(raw_data_path):
    from sklearn.preprocessing import LabelEncoder
    from sklearn.model_selection import train_test_split
    from imblearn.over_sampling import SMOTE
    
    # Drop customerID and convert target
    df_model = df.drop(columns=['customerID'])
    df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})
    
    # Label encode categorical columns
    cat_cols = df_model.select_dtypes(include=['object']).columns.tolist()
    for col in cat_cols:
        df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))
        
    # Split
    X = df_model.drop(columns=['Churn'])
    y = df_model['Churn']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # SMOTE
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    
    print(f"Resampled training set count: {len(X_train_res)}")

### 6.1 Benchmark Classifiers

In [ ]:
if os.path.exists(raw_data_path):
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from xgboost import XGBClassifier
    from sklearn.metrics import classification_report, roc_auc_score
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(random_state=42),
        'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
    }
    
    for name, model in models.items():
        model.fit(X_train_res, y_train_res)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        
        auc = roc_auc_score(y_test, y_proba)
        print(f"\n==================== {name} ====================")
        print(f"ROC-AUC: {auc:.4f}")
        print(classification_report(y_test, y_pred))

## Conclusion
We explored the Telco Customer Churn dataset, cleaned it, engineered 3 key features, and compared models. The models are fully benchmarked, explainable via SHAP, and deployed as a FastAPI REST service and Streamlit web app.